<a href="https://colab.research.google.com/github/ChandrimaGhorai/GeneCraft/blob/main/GeneCraft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
class GeneCraft:
    def __init__(self, reference_dna, sample_dna):
        self.reference_dna = reference_dna.upper()
        self.sample_dna = sample_dna.upper()

        self.codon_table = {
            'ATA':'I', 'ATC':'I', 'ATT':'I', 'ATG':'M',
            'ACA':'T', 'ACC':'T', 'ACG':'T', 'ACT':'T',
            'AAC':'N', 'AAT':'N', 'AAA':'K', 'AAG':'K',
            'AGC':'S', 'AGT':'S', 'AGA':'R', 'AGG':'R',
            'CTA':'L', 'CTC':'L', 'CTG':'L', 'CTT':'L',
            'CCA':'P', 'CCC':'P', 'CCG':'P', 'CCT':'P',
            'CAC':'H', 'CAT':'H', 'CAA':'Q', 'CAG':'Q',
            'CGA':'R', 'CGC':'R', 'CGG':'R', 'CGT':'R',
            'GTA':'V', 'GTC':'V', 'GTG':'V', 'GTT':'V',
            'GCA':'A', 'GCC':'A', 'GCG':'A', 'GCT':'A',
            'GAC':'D', 'GAT':'D', 'GAA':'E', 'GAG':'E',
            'GGA':'G', 'GGC':'G', 'GGG':'G', 'GGT':'G',
            'TCA':'S', 'TCC':'S', 'TCG':'S', 'TCT':'S',
            'TTC':'F', 'TTT':'F', 'TTA':'L', 'TTG':'L',
            'TAC':'Y', 'TAT':'Y', 'TAA':'*', 'TAG':'*',
            'TGC':'C', 'TGT':'C', 'TGA':'*', 'TGG':'W'
        }

        # Amino acid biophysical properties (MW, Net Charge at pH 7, Hydropathy/GRAVY)
        self.aa_properties = {
            'A': {'mw': 71.08, 'charge': 0, 'gravy': 1.8},
            'R': {'mw': 156.20, 'charge': 1, 'gravy': -4.5},
            'N': {'mw': 114.10, 'charge': 0, 'gravy': -3.5},
            'D': {'mw': 115.09, 'charge': -1, 'gravy': -3.5},
            'C': {'mw': 103.14, 'charge': 0, 'gravy': 2.5},
            'Q': {'mw': 128.13, 'charge': 0, 'gravy': -3.5},
            'E': {'mw': 129.12, 'charge': -1, 'gravy': -3.5},
            'G': {'mw': 57.05, 'charge': 0, 'gravy': -0.4},
            'H': {'mw': 137.14, 'charge': 0.1, 'gravy': -3.2},
            'I': {'mw': 113.16, 'charge': 0, 'gravy': 4.5},
            'L': {'mw': 113.16, 'charge': 0, 'gravy': 3.8},
            'K': {'mw': 128.17, 'charge': 1, 'gravy': -3.9},
            'M': {'mw': 131.20, 'charge': 0, 'gravy': 1.9},
            'F': {'mw': 147.18, 'charge': 0, 'gravy': 2.8},
            'P': {'mw': 97.12, 'charge': 0, 'gravy': -1.6},
            'S': {'mw': 87.08, 'charge': 0, 'gravy': -0.8},
            'T': {'mw': 101.11, 'charge': 0, 'gravy': -0.7},
            'W': {'mw': 186.21, 'charge': 0, 'gravy': -0.9},
            'Y': {'mw': 163.18, 'charge': 0, 'gravy': -1.3},
            'V': {'mw': 99.13, 'charge': 0, 'gravy': 4.2},
            '*': {'mw': 0.0, 'charge': 0, 'gravy': 0.0}
        }

    def transcribe(self, dna_seq):
        return dna_seq.replace('T', 'U')

    def translate(self, dna_seq):
        protein = []
        for i in range(0, len(dna_seq) - 2, 3):
            codon = dna_seq[i:i+3]
            amino_acid = self.codon_table.get(codon, 'X')
            if amino_acid == '*':
                break
            protein.append(amino_acid)
        return "".join(protein)

    def validate_start_codon(self, dna_seq):
        positions = []
        for i in range(0, len(dna_seq) - 2, 3):
            if dna_seq[i:i+3] == 'ATG':
                positions.append(i // 3)

        has_start = len(positions) > 0
        starts_with_atg = (0 in positions)

        return {
            'Has Start Codon': has_start,
            'Starts with ATG': starts_with_atg,
            'Positions (0-indexed codon triplet)': positions
        }

    def calculate_dna_metrics(self, dna_seq):
        length = len(dna_seq)
        if length == 0:
            return {'GC Content (%)': 0.0, 'Melting Temp (Tm °C)': 0.0}
        gc_count = dna_seq.count('G') + dna_seq.count('C')
        gc_content = round((gc_count / length) * 100, 2)

        if length < 14:
            tm = (dna_seq.count('A') + dna_seq.count('T')) * 2 + (gc_count * 4)
        else:
            tm = 64.9 + 41 * (gc_count - 16.4) / length

        return {'GC Content (%)': gc_content, 'Melting Temp (Tm °C)': round(tm, 2)}

    def calculate_biophysical_properties(self, protein_seq):
        if not protein_seq:
            return {'Molecular Weight (Da)': 0.0, 'Estimated Net Charge': 0, 'GRAVY Score (Hydropathy)': 0.0}

        total_mw = 18.02
        total_charge = 0
        total_gravy = 0.0

        for aa in protein_seq:
            props = self.aa_properties.get(aa, {'mw': 100.0, 'charge': 0, 'gravy': 0.0})
            total_mw += props['mw']
            total_charge += props['charge']
            total_gravy += props['gravy']

        gravy_score = total_gravy / len(protein_seq) if len(protein_seq) > 0 else 0.0
        return {
            'Molecular Weight (Da)': round(total_mw, 2),
            'Estimated Net Charge': round(total_charge),
            'GRAVY Score (Hydropathy)': round(gravy_score, 3)
        }

    def analyze_mutation(self):
        ref_len = len(self.reference_dna)
        sample_len = len(self.sample_dna)

        if self.reference_dna == self.sample_dna:
            struct_type = "No Mutation"
        elif ref_len == sample_len:
            diffs = sum(1 for a, b in zip(self.reference_dna, self.sample_dna) if a != b)
            struct_type = "SNP" if diffs == 1 else "Multiple Substitutions"
        elif sample_len > ref_len:
            struct_type = "Insertion"
        else:
            struct_type = "Deletion"

        ref_protein = self.translate(self.reference_dna)
        sample_protein = self.translate(self.sample_dna)

        if ref_protein == sample_protein:
            func_effect = "Silent"
        elif len(ref_protein) != len(sample_protein) and abs(sample_len - ref_len) % 3 != 0:
            func_effect = "Frameshift"
        elif len(ref_protein) != len(sample_protein):
            func_effect = "In-frame Indel"
        else:
            if '*' in sample_protein and '*' not in ref_protein:
                func_effect = "Nonsense"
            else:
                func_effect = "Missense"

        return struct_type, func_effect, ref_protein, sample_protein

    def generate_warnings(self, sample_protein):
        warnings = []
        if 'P' in sample_protein:
            warnings.append("Proline residue detected in sample protein; potential alpha-helix breaker or structural rigidity shift.")
        if not warnings:
            warnings.append("No adverse structural or folding red flags triggered.")
        return warnings

    def print_report(self):
        ref_mrna = self.transcribe(self.reference_dna)
        sample_mrna = self.transcribe(self.sample_dna)

        ref_start_info = self.validate_start_codon(self.reference_dna)
        sample_start_info = self.validate_start_codon(self.sample_dna)

        print("\n" + "=" * 65)
        print("GENECRAFT ADVANCED ANALYSIS REPORT")
        print("=" * 65)
        print(f"Reference DNA  : {self.reference_dna}")
        print(f"Sample DNA     : {self.sample_dna}")

        print("\n--- 1. Transcription & Translation ---")
        print(f"Reference mRNA : {ref_mrna}")
        print(f"Sample mRNA    : {sample_mrna}")

        # Check start codons and terminate early if missing
        if not ref_start_info['Has Start Codon']:
            print("  [ERROR] Reference DNA sequence lacks a valid start codon (ATG)! Terminating analysis.")
            print("=" * 65)
            return

        if not sample_start_info['Has Start Codon']:
            print("  [ERROR] Sample DNA sequence lacks a valid start codon (ATG)! Terminating analysis.")
            print("=" * 65)
            return

        ref_protein = self.translate(self.reference_dna)
        sample_protein = self.translate(self.sample_dna)

        print(f"Ref Protein    : {ref_protein}")
        print(f"Sample Protein : {sample_protein}")

        print("\n--- 2. Start Codon Validation ---")
        print(f"Reference Info: {ref_start_info}")
        print(f"Sample Info   : {sample_start_info}")

        ref_dna_metrics = self.calculate_dna_metrics(self.reference_dna)
        sample_dna_metrics = self.calculate_dna_metrics(self.sample_dna)

        ref_profile = self.calculate_biophysical_properties(ref_protein)
        sample_profile = self.calculate_biophysical_properties(sample_protein)

        struct_type, func_effect, _, _ = self.analyze_mutation()

        truncation_loss = 0.0
        ref_p_len = len(ref_protein)
        sample_p_len = len(sample_protein)
        if sample_p_len < ref_p_len and ref_p_len > 0:
            truncation_loss = round(((ref_p_len - sample_p_len) / ref_p_len) * 100, 1)

        print("\n--- 3. DNA Stability & Composition Metrics ---")
        print(f"Reference DNA Metrics: {ref_dna_metrics}")
        print(f"Sample DNA Metrics   : {sample_dna_metrics}")

        print("\n--- 4. Biophysical Property Shifts ---")
        print(f"Reference Protein Profile: {ref_profile}")
        print(f"Sample Protein Profile   : {sample_profile}")
        print(f"ORF Truncation Loss      : {truncation_loss}%")

        print("\n--- 5. Mutation Classification ---")
        print(f"Structural Type  : {struct_type}")
        print(f"Functional Effect: {func_effect}")

        print("\n--- 6. Amino Acid Level Alterations ---")
        if struct_type in ["SNP", "Multiple Substitutions"] and len(ref_protein) == len(sample_protein):
            for idx, (ra, sa) in enumerate(zip(ref_protein, sample_protein)):
                if ra != sa:
                    print(f"  * Position {idx+1}: {ra} -> {sa} substitution")
        else:
            print("  * No direct single-codon changes logged or complex indel pattern.")

        print("\n--- 7. Structural & Biophysical Warnings ---")
        for warning in self.generate_warnings(sample_protein):
            print(f"  * {warning}")
        print("=" * 65)

if __name__ == "__main__":
    print("GeneCraft: Advanced Molecular & Biophysical Pipeline ===")
    ref_input = input("Enter Reference DNA Sequence (e.g., ATGGCCGATTATAGG): ").strip()
    sample_input = input("Enter Sample/Mutated DNA Sequence: ").strip()

    pipeline = GeneCraft(ref_input, sample_input)
    pipeline.print_report()

GeneCraft: Advanced Molecular & Biophysical Pipeline ===
Enter Reference DNA Sequence (e.g., ATGGCCGATTATAGG): atggccgattatagg
Enter Sample/Mutated DNA Sequence: atggcggattatagg

GENECRAFT ADVANCED ANALYSIS REPORT
Reference DNA  : ATGGCCGATTATAGG
Sample DNA     : ATGGCGGATTATAGG

--- 1. Transcription & Translation ---
Reference mRNA : AUGGCCGAUUAUAGG
Sample mRNA    : AUGGCGGAUUAUAGG
Ref Protein    : MADYR
Sample Protein : MADYR

--- 2. Start Codon Validation ---
Reference Info: {'Has Start Codon': True, 'Starts with ATG': True, 'Positions (0-indexed codon triplet)': [0]}
Sample Info   : {'Has Start Codon': True, 'Starts with ATG': True, 'Positions (0-indexed codon triplet)': [0]}

--- 3. DNA Stability & Composition Metrics ---
Reference DNA Metrics: {'GC Content (%)': 46.67, 'Melting Temp (Tm °C)': 39.21}
Sample DNA Metrics   : {'GC Content (%)': 46.67, 'Melting Temp (Tm °C)': 39.21}

--- 4. Biophysical Property Shifts ---
Reference Protein Profile: {'Molecular Weight (Da)': 654.77, 'E